# Non-Life Insurance Pricing with GLM

I wanted to understand how insurers price motor policies.

This notebook models claim frequency on French motor insurance data
(`freMTPL2freq`) using a Generalized Linear Model (GLM), a standard
approach in actuarial pricing.

The objective is not to build a complete pricing system, but to understand
the data, estimate claim frequency, interpret the main effects, and evaluate
the model on unseen data.


## 1. Imports and setup


In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid")


## 2. Load the data

The dataset contains individual motor insurance policies.
`ClaimNb` is the observed number of claims and `Exposure` represents
the fraction of a year for which the policy was observed.

For frequency modelling, exposure is important because two policies
may have been observed for different lengths of time.


In [ ]:
# Find the project root whether the notebook is opened from
# the project root or from the notebooks/ folder.
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

path = ROOT / "data" / "raw" / "freMTPL2freq.csv"

if not path.exists():
    raise FileNotFoundError(f"Dataset not found: {path}")

df = pd.read_csv(path)

print(f"Dataset: {df.shape[0]:,} policies, {df.shape[1]} variables")
df.head()


## 3. First look at the data


In [ ]:
print("Missing values:")
print(df.isnull().sum())

print("\nClaim distribution:")
print(df["ClaimNb"].value_counts().sort_index())

print("\nNumerical summary:")
print(df[["ClaimNb", "Exposure"]].describe().round(4))


Most policies have no claim during the observation period, which is
typical for motor insurance frequency data.

The model uses only observations with positive exposure because the
logarithm of exposure is used as an offset.


In [ ]:
df = df[df["Exposure"] > 0].copy()

print(f"Policies used: {len(df):,}")
print(f"Mean claims: {df['ClaimNb'].mean():.4f}")
print(f"Mean exposure: {df['Exposure'].mean():.4f}")


## 4. Claim frequency

A simple descriptive measure is the total number of claims divided by
the total exposure. This gives the observed average claim frequency
per year of exposure.


In [ ]:
observed_frequency = df["ClaimNb"].sum() / df["Exposure"].sum()

print(f"Observed claim frequency: {observed_frequency:.4f} claims per year")


## 5. Exploratory analysis

Before fitting a model, I look at a few variables that are directly
related to the policyholder or the vehicle.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

df.groupby("DrivAge")["ClaimNb"].mean().plot(ax=axes[0])
axes[0].set_title("Mean claims by driver age")
axes[0].set_xlabel("Driver age")
axes[0].set_ylabel("Mean claim count")

df.groupby("BonusMalus")["ClaimNb"].mean().plot(ax=axes[1])
axes[1].set_title("Mean claims by Bonus-Malus")
axes[1].set_xlabel("Bonus-Malus")
axes[1].set_ylabel("Mean claim count")

df["Exposure"].hist(ax=axes[2], bins=50)
axes[2].set_title("Exposure distribution")
axes[2].set_xlabel("Exposure (years)")
axes[2].set_ylabel("Number of policies")

plt.tight_layout()
plt.show()


## 6. Prepare variables for the GLM

The model uses a mixture of numerical and categorical variables.

Categorical variables are converted to indicator variables. One category
is kept as the reference category to avoid perfect multicollinearity.

The exposure is not used as an ordinary explanatory variable. Instead,
its logarithm is used as an **offset**, so the model estimates claim
frequency rather than simply the number of claims.


`BonusMalus` is centered at 50 so that the model intercept corresponds
to a more meaningful reference level.

The centered variable is used only for the model; the original variable
is kept for descriptive analysis.


In [ ]:
df["BonusMalus_c"] = df["BonusMalus"] - 50

target = "ClaimNb"
offset = np.log(df["Exposure"])

numeric_vars = ["VehPower", "VehAge", "DrivAge", "BonusMalus_c", "Density"]
categorical_vars = ["Area", "VehBrand", "VehGas", "Region"]

X = pd.get_dummies(
    df[numeric_vars + categorical_vars],
    columns=categorical_vars,
    drop_first=True,
    dtype=float
)

y = df[target].astype(float)

print(f"Model matrix: {X.shape[0]:,} rows x {X.shape[1]} variables")


## 7. Train/test split

I keep a separate test set so that the final evaluation is performed
on observations that were not used to estimate the model.


In [ ]:
X_train, X_test, y_train, y_test, off_train, off_test = train_test_split(
    X, y, offset,
    test_size=0.20,
    random_state=42
)

print(f"Training observations: {len(X_train):,}")
print(f"Test observations: {len(X_test):,}")


## 8. Fit the Poisson GLM

For claim frequency, a Poisson GLM is a natural starting point because
the response is a non-negative claim count.

The logarithm of exposure is included as an offset:

log(E[ClaimNb]) = log(Exposure) + Xβ

Equivalently, the model describes the expected claim frequency through
the explanatory variables while accounting for different exposure times.


In [ ]:
model = sm.GLM(
    y_train,
    sm.add_constant(X_train),
    family=sm.families.Poisson(),
    offset=off_train
)

result = model.fit()

print(result.summary())


### Poisson dispersion check

For a Poisson model, the variance is theoretically linked to the mean.
A Pearson dispersion value clearly above 1 can indicate overdispersion,
which would motivate comparing the model with a Negative Binomial approach.


In [ ]:
pearson_dispersion = result.pearson_chi2 / result.df_resid

print(f"Pearson dispersion: {pearson_dispersion:.3f}")
if pearson_dispersion > 1:
    print("The Poisson model shows some evidence of overdispersion; this deserves further investigation.")
else:
    print("The Pearson dispersion is not above 1 in this sample.")


## 9. Interpret the coefficients

Because the Poisson GLM uses a log link, exponentiating a coefficient
gives a multiplicative effect on the expected claim rate, holding the
other variables constant.

For example, an exponentiated coefficient of 1.05 corresponds to an
estimated 5% higher expected rate for a one-unit increase in that
variable, all else equal.


In [ ]:
coef_table = pd.DataFrame({
    "coef": result.params,
    "rate_ratio": np.exp(result.params)
})

coef_table = coef_table.sort_values("rate_ratio", ascending=False)

coef_table.head(15)


### Focus on Bonus-Malus

Bonus-Malus is particularly relevant from an insurance perspective
because it reflects the driver’s claims history and can therefore be
associated with the expected claim frequency.

The model coefficient should be interpreted as an association estimated
from this dataset, not as a causal effect.


In [ ]:
if "BonusMalus_c" in result.params.index:
    bm_coef = result.params["BonusMalus_c"]
    bm_ratio = np.exp(bm_coef)

    print(f"Bonus-Malus coefficient (centered): {bm_coef:.6f}")
    print(f"Rate ratio for +1 Bonus-Malus point: {bm_ratio:.4f}")
    print(f"Approximate percentage change: {(bm_ratio - 1) * 100:.2f}%")
else:
    print("Bonus-Malus coefficient not found.")


## 10. Evaluate the model

Predictions are produced on the test set using the same exposure offset.

RMSE measures the average prediction error on the claim-count scale.
The Gini coefficient gives an additional measure of ranking/discrimination,
which is useful in insurance pricing contexts.


In [ ]:
y_pred = result.predict(
    sm.add_constant(X_test),
    offset=off_test
)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))

def exposure_weighted_gini(y_true, y_pred_count, exposure):
    data = pd.DataFrame({
        "claims": np.asarray(y_true),
        "pred_count": np.asarray(y_pred_count),
        "exposure": np.asarray(exposure),
    })

    data["pred_frequency"] = data["pred_count"] / data["exposure"]
    data = data.sort_values("pred_frequency", ascending=False)

    total_claims = data["claims"].sum()
    total_exposure = data["exposure"].sum()

    if total_claims == 0 or total_exposure == 0:
        return np.nan

    cumulative_claims = data["claims"].cumsum() / total_claims
    cumulative_exposure = data["exposure"].cumsum() / total_exposure

    x = np.concatenate(([0.0], cumulative_exposure.to_numpy()))
    y = np.concatenate(([0.0], cumulative_claims.to_numpy()))

    return 2 * np.trapezoid(y, x) - 1

gini_score = exposure_weighted_gini(
    y_test.to_numpy(),
    y_pred.to_numpy(),
    np.exp(off_test.to_numpy())
)

print(f"RMSE : {rmse:.6f}")
print(f"Exposure-weighted Gini : {gini_score:.4f}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].scatter(y_test, y_pred, alpha=0.25, s=8)
max_value = max(y_test.max(), y_pred.max())
axes[0].plot([0, max_value], [0, max_value], "--", linewidth=1.5)
axes[0].set_xlabel("Actual claim count")
axes[0].set_ylabel("Predicted claim count")
axes[0].set_title("Predicted vs actual")

residuals = y_test.to_numpy() - y_pred.to_numpy()
axes[1].hist(residuals, bins=50, edgecolor="white", alpha=0.8)
axes[1].axvline(0, linestyle="--", linewidth=1.5)
axes[1].set_xlabel("Residuals")
axes[1].set_title("Residual distribution")

plt.tight_layout()
plt.show()


## 11. What I learned

This first model shows how a classical actuarial GLM can be used to
estimate motor insurance claim frequency.

The main points are:

- exposure must be taken into account when modelling claim counts;
- categorical policy characteristics can be included through indicator
  variables;
- the Poisson log-link makes coefficients easy to interpret through
  multiplicative rate ratios;
- model performance should be assessed on data that were not used for
  estimation;
- the dispersion check helps assess whether the Poisson variance assumption
  is reasonable for this dataset.

The results also show that the available variables do not explain all
the variation in claim frequency. A natural next step would be to compare
this Poisson model with a Negative Binomial model, and to study possible
interactions or additional spatial information.

A complete insurance pricing model would also require a severity model
for the cost of claims, in addition to the frequency model studied here.
